### Functions

In [1]:
def load_session_data(subject, date):
    """Load all data for a given subject and date"""
    import sys
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_utils.NeuralDataLoader import NeuralDataLoader, Dots3DMPConfig
    
    # Load session
    loader = NeuralDataLoader()
    loader.load_session(subject, date)
    config = Dots3DMPConfig(subject)

    # spike data (unit, trial, time)
    stimOn_spikes = loader.get_spike_data(alignment='stimOn', good_units_only=True, good_trials_only=True)
    saccOnset_spikes = loader.get_spike_data(alignment='saccOnset', good_units_only=True, good_trials_only=True)
    postTargHold_spikes = loader.get_spike_data(alignment='postTargHold', good_units_only=True, good_trials_only=True)
    tuning_spikes = loader.get_tuning_data(good_units_only=True, good_trials_only=True)

    # behavioral data
    behavior_dots3DMP = loader.get_behavioral_data(task='dots3DMP', good_trials_only=True, cal_mean_RT=True)
    behavior_tuning = loader.get_behavioral_data(task='tuning', good_trials_only=True)
    behavior_converted = config.convert_behavioral_data(behavior_dots3DMP, task='dots3DMP')
    behavior_tuning_converted = config.convert_behavioral_data(behavior_tuning, task='tuning')

    # Unit Info
    unit_info = loader.get_unit_info(good_units_only=True)
    MST_units = loader.get_units_by_area(unit_info, area_name='MST')
    VPS_units = loader.get_units_by_area(unit_info, area_name='VPS')
    MT_units = loader.get_units_by_area(unit_info, area_name='MT')
    dual_units = loader.get_units_by_area(unit_info, area_name='dual')

    # Time Info
    time_info = config.get_time_Info('dots3DMP')
    time_info_tuning = config.get_time_Info('tuning')
    time_axes_dots3DMP = config.get_time_axes('dots3DMP')
    time_axes_tuning = config.get_time_axes('tuning')

    # Prepare data
    spikes_data = {
        'stimOn': stimOn_spikes,
        'saccOnset': saccOnset_spikes,
        'postTargHold': postTargHold_spikes,
    }

    tuning_spikes_data = {'stimOn': tuning_spikes}
    
    units_data = {
        'MST': MST_units,
        'VPS': VPS_units,
        'MT': MT_units,
        'dual': dual_units
    }
    
    return {
        'loader': loader,
        'config': config,
        'spikes_data': spikes_data,
        'tuning_spikes_data': tuning_spikes_data,
        'behavior_converted': behavior_converted,
        'behavior_tuning_converted': behavior_tuning_converted,
        'unit_info': unit_info,
        'units_data': units_data,
        'time_axes_dots3DMP': time_axes_dots3DMP,
        'time_axes_tuning': time_axes_tuning,  
        'time_info': time_info,
        'time_info_tuning': time_info_tuning,
    }

In [2]:
def pool_all_sessions(session_list, alignment='stimOn', save_pooled=True, is_tuning=True, is_RT=False):
    """
    Pool all sessions for neural feature analysis.
    
    Parameters:
    -----------
    session_list : list of tuples
        Each tuple contains (subject, date, abs_pos, rel_pos, guidetube_depth, rel_guidetube_depth)
    alignment : str
        'stimOn' or 'saccOnset'
    save_pooled : bool
        Whether to save the pooled data
    is_tuning : bool
        True for tuning task, False for regular task
    is_RT : bool
        True to use RT-based time window, False for sliding windows (tuning) or pre-saccade (regular)
    
    Modes:
    ------
    Mode 1: is_tuning=True,  is_RT=False -> Tuning with sliding windows (alignment='stimOn')
    Mode 2: is_tuning=True,  is_RT=True  -> Tuning with RT window (alignment='stimOn')
    Mode 3: is_tuning=False, is_RT=False -> Regular pre-choice (alignment='saccOnset')
    Mode 4: is_tuning=False, is_RT=True  -> Regular with RT window (alignment='stimOn')
    """
    import sys
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_single_neurons.neural_feature_analyzer import NeuralFeatureAnalyzer
    import pandas as pd
    import os

    all_features = []
    failed_sessions = []
    
    # Determine task type and save directory based on is_tuning and is_RT
    if is_tuning:
        save_dir = r'D:\Neural-Pipeline\results\analysis_single_neurons\dots3DMPtuning_neuralfeatures'
        if is_RT:
            task_type = 'tuning_RT'
            print("Mode 2: TUNING task with RT window")
        else:
            task_type = 'tuning'
            print("Mode 1: TUNING task with sliding windows")
    else:
        save_dir = r'D:\Neural-Pipeline\results\analysis_single_neurons\dots3DMP_neuralfeatures'
        if is_RT:
            task_type = 'regular_RT'
            print("Mode 4: REGULAR task with RT window")
        else:
            task_type = 'regular'
            print("Mode 3: REGULAR task with pre-saccade window")
    
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"Processing {len(session_list)} sessions...")
    print(f"  Alignment: {alignment}")
    print(f"  is_tuning: {is_tuning}")
    print(f"  is_RT: {is_RT}")
    print(f"  Task type: {task_type}")
    print("=" * 60)
    
    for i, (subject, date, abs_pos, rel_pos, guidetube_depth, rel_guidetube_depth) in enumerate(session_list):
        try:
            print(f"\nProcessing session {i+1}/{len(session_list)}: {subject} {date}")
            print(f"  Position: absolute {abs_pos}, relative {rel_pos}")
            print(f"  Guidetube depth: {guidetube_depth}, relative: {rel_guidetube_depth:.3f} mm")
            print(f"  Task type: {task_type} (alignment: {alignment})")
            
            # Load session data
            session_data = load_session_data(subject, date)
            
            # Get mean_RT if needed
            if is_RT:
                mean_RT = session_data['behavior_converted']['mean_RT']
                print(f"  Mean RT loaded: {mean_RT}")
            else:
                mean_RT = None

            # Create analyzer with correct parameters
            analyzer = NeuralFeatureAnalyzer(
                session_data, 
                subject, 
                date, 
                alignment=alignment,
                session_position=abs_pos, 
                relative_position=rel_pos,
                session_guidetube_depth=guidetube_depth,
                relative_guidetube_depth=rel_guidetube_depth,
                is_tuning=is_tuning,
                mean_RT=mean_RT
            )
            
            # Analyze all units
            analyzer.analyze_all_units()
            
            # Get features
            session_features = analyzer.get_features_dataframe()
            
            if len(session_features) > 0:
                all_features.append(session_features)
                
                # Quick summary
                n_MST = len(session_features[session_features['area'] == 'MST'])
                n_VPS = len(session_features[session_features['area'] == 'VPS'])
                n_dual = len(session_features[session_features['area'] == 'dual'])
                n_unknown = len(session_features[session_features['area'] == 'unknown'])
                mean_rate = session_features['overall_firing_rate'].mean()
                
                print(f"  ✓ Added {len(session_features)} units")
                print(f"    MST: {n_MST}, VPS: {n_VPS}, dual: {n_dual}, unknown: {n_unknown}")
                print(f"    Mean firing rate: {mean_rate:.2f} Hz")
                
            else:
                print(f"  ⚠ No units found in session")
            
        except Exception as e:
            print(f"  ✗ Failed to process {subject} {date}: {e}")
            import traceback
            traceback.print_exc()
            failed_sessions.append((subject, date))
            continue
    
    # Combine and save results
    if all_features:
        pooled_features = pd.concat(all_features, ignore_index=True)
        
        print("\n" + "=" * 60)
        print("POOLING SUMMARY")
        print("=" * 60)
        print(f"Successfully pooled {len(pooled_features)} units from {len(all_features)}/{len(session_list)} sessions")
        print(f"Task type: {task_type} (alignment: {alignment})")
        
        if failed_sessions:
            print(f"Failed sessions: {[f'{s}_{d}' for s, d in failed_sessions]}")
        
        if save_pooled:
            # Create filename based on task type and alignment
            pooled_filename = f"zarya_pooled_neural_features_{task_type}_{alignment}.csv"
            pooled_path = os.path.join(save_dir, pooled_filename)
            pooled_features.to_csv(pooled_path, index=False)
            print(f"\nDataset saved to: {pooled_path}")
        
        # Final summary
        area_counts = pooled_features['area'].value_counts()
        print(f"\nFinal dataset: {len(pooled_features)} units")
        for area, count in area_counts.items():
            print(f"  {area}: {count} units ({count/len(pooled_features)*100:.1f}%)")
        
        # Show statistics for tuning sliding windows
        if is_tuning and not is_RT:
            print(f"\nTuning sliding window statistics:")
            
            for modality in ['ves', 'vis', 'comb']:
                threshold_col = f'neurometric_{modality}_thresholds'
                if threshold_col in pooled_features.columns:
                    import ast
                    all_thresholds = []
                    for thresh_list_str in pooled_features[threshold_col]:
                        try:
                            thresh_list = ast.literal_eval(thresh_list_str)
                            all_thresholds.extend([t for t in thresh_list if not np.isnan(t)])
                        except:
                            pass
                    
                    if len(all_thresholds) > 0:
                        print(f"  {modality.upper()} neurometric thresholds: {len(all_thresholds)} valid measurements, "
                              f"mean = {np.mean(all_thresholds):.2f}°")
        
        return pooled_features
    else:
        print("❌ No sessions were successfully processed!")
        return pd.DataFrame()

### Main code

In [3]:
### Main code
subject = 'zarya'
dates = ['20250306', '20250411', '20250417', '20250501', '20250523', '20250602', '20250702', '20250710']
center = (4, 4, 22)  # Center position (x, y, z), z = corrected guide tube length in mm

guide_tube_depths = [
    (22, 16.745),  # 20250306
    (26, 15.000),  # 20250411
    (26, 11.974),  # 20250417
    (25, 11.500),  # 20250501
    (24, 13.710),  # 20250523
    (24, 11.975),  # 20250602
    (22, 13.800),  # 20250702
    (23, 13.000)   # 20250710
]

# Position for each session
positions = [
    (4, 4),  # 20250306
    (4, 5),  # 20250411
    (4, 3),  # 20250417
    (5, 4),  # 20250501
    (4, 4),  # 20250523
    (4, 5),  # 20250602
    (4, 3),  # 20250702
    (5, 4)   # 20250710
]

# Calculate relative positions and depths from center
relative_guide_tube_depths = [(depth[0] + depth[1] - center[2]) for depth in guide_tube_depths]
relative_positions = [(pos[0] - center[0], pos[1] - center[1]) for pos in positions]

# Create session list with all parameters
session_list = [(subject, dates[i], positions[i], relative_positions[i], guide_tube_depths[i], relative_guide_tube_depths[i]) 
                for i in range(len(dates))]

# ============================================================================
# MODE 1: Tuning task with sliding windows (alignment='stimOn')
# ============================================================================
# print("="*80)
# print("MODE 1: Processing TUNING data with SLIDING WINDOWS...")
# print("="*80)
# pooled_tuning_data = pool_all_sessions(
#     session_list, 
#     alignment='stimOn', 
#     is_tuning=True, 
#     is_RT=False
# )

# ============================================================================
# MODE 2: Tuning task with RT window (alignment='stimOn')
# ============================================================================
print("\n" + "="*80)
print("MODE 2: Processing TUNING data with RT WINDOW...")
print("="*80)
pooled_tuning_rt_data = pool_all_sessions(
    session_list, 
    alignment='stimOn', 
    is_tuning=True, 
    is_RT=True
)

# # # ============================================================================
# # # MODE 4: Regular task with pre-saccade window (alignment='saccOnset')
# # # ============================================================================
print("\n" + "="*80)
print("MODE 4: Processing REGULAR TASK data with PRE-SACCADE WINDOW...")
print("="*80)
pooled_regular_data = pool_all_sessions(
    session_list, 
    alignment='saccOnset', 
    is_tuning=False, 
    is_RT=False
)

# ============================================================================
# MODE 3: Regular task with RT window (alignment='stimOn')
# ============================================================================
print("\n" + "="*80)
print("MODE 3: Processing REGULAR TASK data with RT WINDOW...")
print("="*80)
pooled_regular_rt_data = pool_all_sessions(
    session_list, 
    alignment='stimOn', 
    is_tuning=False, 
    is_RT=True
)

# # ============================================================================
# # FINAL SUMMARY
# # ============================================================================
# print("\n" + "="*80)
# print("FINAL SUMMARY")
# print("="*80)

# datasets = [
#     # ("Mode 1: Tuning (sliding windows)", pooled_tuning_data),
#     ("Mode 2: Tuning (RT window)", pooled_tuning_rt_data),
#     ("Mode 3: Regular (pre-saccade)", pooled_regular_data),
#     ("Mode 4: Regular (RT window)", pooled_regular_rt_data),
# ]

# for name, data in datasets:
#     if len(data) > 0:
#         print(f"\n{name}: {len(data)} units")
#         areas = data['area'].value_counts()
#         for area, count in areas.items():
#             print(f"  {area}: {count}")
#     else:
#         print(f"\n{name}: No data")

print("\n✅ All 4 modes complete!")


MODE 2: Processing TUNING data with RT WINDOW...
Mode 2: TUNING task with RT window
Processing 8 sessions...
  Alignment: stimOn
  is_tuning: True
  is_RT: True
  Task type: tuning_RT

Processing session 1/8: zarya 20250306
  Position: absolute (4, 4), relative (0, 0)
  Guidetube depth: (22, 16.745), relative: 16.745 mm
  Task type: tuning_RT (alignment: stimOn)
Loaded dots3DMP data: zarya20250306dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250306dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya
  Mean RT loaded: {'mod1_coh1': 0.6370715564556337, 'mod2_coh1': 0.8803190225965521, 'mod2_coh2': 0.7382467119881276, 'mod3_coh1': 0.6912505726237875, 'mod3_coh2': 0.6617574031536397}
Debug: Session 20250306 at position (4, 4) (relative: (0, 0))
Debug: Guidetube depth: (22, 16.745), relative: 16.745 mm
Debug: Mode 2: Tuning with mean RT-based fixed windows
Debug: Alignment: stimOn
Debug: Mean RTs provided: {'mod1_coh1': 0.6370715564556337, 'mod2_coh1': 